## this is complete Transformer component used in BERT-style models
### took help from chat GPT

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Self-Attention Head
class BertAttentionHead(nn.Module):

    def __init__(
        self,
        embed_dim,
        head_size
    ):

        super().__init__()

        self.query = nn.Linear(
            embed_dim,
            head_size,
            bias=False
        )

        self.key = nn.Linear(
            embed_dim,
            head_size,
            bias=False
        )

        self.value = nn.Linear(
            embed_dim,
            head_size,
            bias=False
        )

    def forward(self, x):

        Q = self.query(x)

        K = self.key(x)

        V = self.value(x)

        scores = Q @ K.transpose(-2,-1)

        scores = (
            scores /
            (K.shape[-1] ** 0.5)
        )

        weights = F.softmax(
            scores,
            dim=-1
        )

        out = weights @ V

        return out

In [3]:
# Multi Head Attention
class BertMultiHeadAttention(
    nn.Module
):

    def __init__(
        self,
        embed_dim,
        num_heads
    ):

        super().__init__()

        head_size = (
            embed_dim //
            num_heads
        )

        self.heads = nn.ModuleList(

            [

                BertAttentionHead(
                    embed_dim,
                    head_size
                )

                for _ in range(
                    num_heads
                )

            ]

        )

        self.proj = nn.Linear(
            embed_dim,
            embed_dim
        )

    def forward(self, x):

        out = torch.cat(

            [

                head(x)

                for head in self.heads

            ],

            dim=-1

        )

        out = self.proj(out)

        return out

In [4]:
# Feed Forward Network
class FeedForward(
    nn.Module
):

    def __init__(
        self,
        embed_dim
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                embed_dim,
                4 * embed_dim
            ),

            nn.GELU(),

            nn.Linear(
                4 * embed_dim,
                embed_dim
            )

        )

    def forward(
        self,
        x
    ):

        return self.net(x)

In [5]:
# Encoder Block
class BertEncoderBlock(
    nn.Module
):

    def __init__(
        self,
        embed_dim,
        num_heads
    ):

        super().__init__()

        self.attn = (
            BertMultiHeadAttention(
                embed_dim,
                num_heads
            )
        )

        self.ffwd = FeedForward(
            embed_dim
        )

        self.ln1 = nn.LayerNorm(
            embed_dim
        )

        self.ln2 = nn.LayerNorm(
            embed_dim
        )

    def forward(
        self,
        x
    ):

        x = self.ln1(
            x +
            self.attn(x)
        )

        x = self.ln2(
            x +
            self.ffwd(x)
        )

        return x

In [6]:
#Test
B = 2
T = 10
C = 768

x = torch.randn(
    B,
    T,
    C
)

block = BertEncoderBlock(
    embed_dim=768,
    num_heads=12
)

out = block(x)

print(out.shape)

torch.Size([2, 10, 768])
